In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def path(name):
    return f'/content/drive/MyDrive/nll/{name}'

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/nll'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/nll'

In [ ]:
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'training_curves_7cat.png', 'confusion_7cat.png', 'confusion_zeroshot_7cat.png', 'confusion_7cat_eval.png', 'zeroshot_vs_finetuned_6cat.png', 'confusion_6cat_zeroshot.png', 'confusion_6cat_v2_zeroshot.png', 'zeroshot_vs_finetuned_6cat_v2.png', 'training_curves_6cat_v3.png', 'confusion_6cat_v3_finetuned.png', 'siglip_6cat', 'data_6cat_large.zip', 'siglip_6cat_large', 'Copy of best_siglip.pth', 'best_siglip_festival.pth', 'data_pseudo.zip', 'Copy of data_pseudo.zip', 'siglip_6cat_pseudo', 'clip_6cat_festival', 'phototrip_3dgs', 'Copy of 1강_basic mission_answer.gdoc', '1강_advanced mission_answer.gdoc', '1강_basic mission_answer_sheet.pdf.gdoc', '2강_basic mission_answer_sheet.pdf.gdoc', '3강_basic mission_answer_sheet.pdf.gdoc', '6강_basic mission_question_sheet.pdf.gdoc', '4강_basic mission_question_sheet.pdf.gdoc', '5강_basic mission_answer_sheet.pdf.gdoc', 'train.csv', 'val.csv', 'test.csv']


In [ ]:
import glob
print(glob.glob('/content/drive/MyDrive/**/train.csv', recursive=True))

['/content/drive/MyDrive/train.csv']


In [ ]:
def path(name):
    return f'/content/drive/MyDrive/{name}'

print(os.path.exists(path('train.csv')), os.path.exists(path('val.csv')), os.path.exists(path('test.csv')))

True True True


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, log_loss,
                             confusion_matrix, precision_score, recall_score)

CLASSES = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
           'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
           'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
           'gpt-6-astra','kimi-k3','qwen3.8-max']
PREFIX = {'claude':'Anthropic','gemini':'Google','glm':'ZAI','gpt':'OpenAI','kimi':'Moonshot','qwen':'Qwen'}
COMPANIES = sorted(PREFIX.values())
M2C = np.array([COMPANIES.index(PREFIX[m.split('-')[0].rstrip('0123456789.')]) for m in CLASSES])
MULTI = np.bincount(M2C, minlength=6) > 1

def to_company(P):
    Q = np.zeros((len(P), 6))
    for m, c in enumerate(M2C):
        Q[:, c] += P[:, m]
    return Q

def ece(y, P, n_bins=15):
    conf, hit = P.max(1), P.argmax(1) == y
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    return float(sum((bins == b).mean() * abs(hit[bins == b].mean() - conf[bins == b].mean())
                     for b in range(n_bins) if (bins == b).any()))

def score(y, P):
    pred, yc, Pc = P.argmax(1), M2C[y], to_company(P)
    predc = Pc.argmax(1)
    mask = MULTI[yc] & (M2C[pred] == yc)
    return {
        'model_BA': balanced_accuracy_score(y, pred),
        'model_Precision': precision_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_Recall': recall_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_F1': f1_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_F1w': f1_score(y, pred, average='weighted', labels=range(18), zero_division=0),
        'model_Acc': accuracy_score(y, pred),
        'model_LogLoss': log_loss(y, P, labels=range(18)),
        'model_ECE': ece(y, P),
        'sibling_Acc': (pred[mask] == y[mask]).mean() if mask.any() else np.nan,
        'company_BA': balanced_accuracy_score(yc, predc),
        'company_BA_argmax': balanced_accuracy_score(yc, M2C[pred]),
        'company_Precision': precision_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_Recall': recall_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_F1': f1_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_F1w': f1_score(yc, predc, average='weighted', labels=range(6), zero_division=0),
        'company_Acc': accuracy_score(yc, predc),
        'company_LogLoss': log_loss(yc, Pc, labels=range(6)),
        'company_ECE': ece(yc, Pc),
    }

def load_data(train_path, val_path, test_path):
    parts = {'train': train_path, 'val': val_path, 'test': test_path}
    data = pd.concat([pd.read_csv(p).assign(split=k) for k, p in parts.items()], ignore_index=True)
    n = data.split.value_counts().to_dict()
    assert n == {'train': 1889, 'test': 414, 'val': 396}, f'A의 분할 파일이 아님: {n}'
    assert data.id.is_unique and data.groupby('prompt_id').split.nunique().eq(1).all(), '분할 오류'
    return data[['id', 'prompt_id', 'model', 'split']]

def load_run(paths, data, split='val', open_test=False):
    if split not in ('val', 'test'):
        raise ValueError("split은 'val' 또는 'test'만 가능")
    if split == 'test' and not open_test:
        raise PermissionError('test는 정해진 시점에만 open_test=True로 연다')
    paths = [paths] if isinstance(paths, str) else list(paths)
    meta = data[data.split == split].sort_values('id').reset_index(drop=True)
    Ps = []
    for p in paths:
        df = pd.read_csv(p)
        assert list(df.columns) == ['id'] + CLASSES, f'{p}: 컬럼은 id + CLASSES(알파벳순)'
        assert df.id.is_unique and set(df.id) == set(meta.id), f'{p}: id가 평가 대상과 불일치'
        P = df.set_index('id').loc[meta.id, CLASSES].to_numpy(float)
        assert np.isfinite(P).all() and (P >= 0).all(), f'{p}: NaN 또는 음수 확률'
        assert np.allclose(P.sum(1), 1, atol=1e-3), f'{p}: 행 합이 1이 아님'
        Ps.append(P / P.sum(1, keepdims=True))
    y = meta.model.map({m: i for i, m in enumerate(CLASSES)}).to_numpy()
    return {'split': split, 'meta': meta, 'P': np.stack(Ps), 'y': y}

def evaluate(run):
    rows = pd.DataFrame([score(run['y'], P) for P in run['P']])
    return pd.DataFrame({'mean': rows.mean(), 'std': rows.std(ddof=1) if len(rows) > 1 else 0.0})

def _boot(meta, n_boot, seed):
    rng = np.random.default_rng(seed)
    groups = list(meta.groupby('prompt_id').indices.values())
    for _ in range(n_boot):
        yield np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])

def _ba(y, pred, n):
    tot = np.bincount(y, minlength=n)
    hit = np.bincount(y, weights=pred == y, minlength=n)
    return (hit[tot > 0] / tot[tot > 0]).mean()

def _fast(y, Ps):
    return np.mean([[_ba(y, P.argmax(1), 18), _ba(M2C[y], to_company(P).argmax(1), 6)] for P in Ps], 0)

def bootstrap_ci(run, n_boot=1000, seed=0):
    b = np.array([_fast(run['y'][i], run['P'][:, i]) for i in _boot(run['meta'], n_boot, seed)])
    lo, hi = np.percentile(b, [2.5, 97.5], axis=0)
    return pd.DataFrame({'ci_low': lo, 'ci_high': hi}, index=['model_BA', 'company_BA'])

def compare(base, new, n_boot=1000, seed=0):
    assert base['split'] == new['split'] and base['meta'].id.equals(new['meta'].id), '평가 대상이 다름'
    d = np.array([_fast(base['y'][i], new['P'][:, i]) - _fast(base['y'][i], base['P'][:, i])
                  for i in _boot(base['meta'], n_boot, seed)])
    lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
    out = pd.DataFrame({'diff_mean': d.mean(0), 'ci_low': lo, 'ci_high': hi}, index=['model_BA', 'company_BA'])
    out['improved'] = out.ci_low > 0
    out['worse'] = out.ci_high < 0
    return out

def confusion(run, level='model'):
    y, names = (run['y'], CLASSES) if level == 'model' else (M2C[run['y']], COMPANIES)
    pred = (lambda P: P.argmax(1)) if level == 'model' else (lambda P: to_company(P).argmax(1))
    Ms = [confusion_matrix(y, pred(P), labels=range(len(names)), normalize='true') for P in run['P']]
    return pd.DataFrame(np.mean(Ms, 0), index=names, columns=names)

def leaderboard(runs):
    return pd.DataFrame({k: evaluate(r).apply(lambda s: f"{s['mean']:.4f} ± {s['std']:.4f}", axis=1)
                         for k, r in runs.items()}).T

def template_table(val_run, test_run, level='model'):
    rows = {'Accuracy': 'Acc', 'Precision (Macro)': 'Precision', 'Recall (Macro) = BA': 'Recall',
            'F1-score (Macro)': 'F1', 'F1-score (Weighted)': 'F1w', 'Log loss': 'LogLoss', 'ECE': 'ECE'}
    if level == 'model':
        rows['형제 정확도'] = None
    dv, te = evaluate(val_run), evaluate(test_run)
    key = lambda v: 'sibling_Acc' if v is None else f'{level}_{v}'
    return pd.DataFrame({'Validation': [f"{dv.loc[key(v), 'mean']:.4f} ± {dv.loc[key(v), 'std']:.4f}" for v in rows.values()],
                         'Test': [f"{te.loc[key(v), 'mean']:.4f}" for v in rows.values()]}, index=list(rows))

In [ ]:
def save_pred(ids, P, labels, name):
    df = pd.DataFrame(P, columns=list(labels))[CLASSES]
    df.insert(0, 'id', list(ids))
    df.to_csv(name, index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tr, va, te = [pd.read_csv(path(f'{n}.csv')) for n in ['train', 'val', 'test']]
vec = TfidfVectorizer(analyzer='char', ngram_range=(2, 6), sublinear_tf=True,
                      min_df=3, max_features=400000)
clf = LogisticRegression(C=10, max_iter=3000).fit(vec.fit_transform(tr.text), tr.model)

save_pred(va.id, clf.predict_proba(vec.transform(va.text)), clf.classes_, path('val_BASE_tfidfchar_v2.csv'))
save_pred(te.id, clf.predict_proba(vec.transform(te.text)), clf.classes_, path('test_BASE_tfidfchar_v2.csv'))
print('저장 완료')

저장 완료


In [ ]:
data = load_data(path('train.csv'), path('val.csv'), path('test.csv'))

base = load_run(path('val_BASE_tfidfchar_v2.csv'), data)
print(evaluate(base).round(4))

base_test = load_run(path('test_BASE_tfidfchar_v2.csv'), data, split='test', open_test=True)
print(evaluate(base_test).round(4))

print(template_table(base, base_test, level='company'))
print(template_table(base, base_test, level='model'))

                     mean  std
model_BA           0.2904  0.0
model_Precision    0.2949  0.0
model_Recall       0.2904  0.0
model_F1           0.2810  0.0
model_F1w          0.2810  0.0
model_Acc          0.2904  0.0
model_LogLoss      2.3244  0.0
model_ECE          0.1401  0.0
sibling_Acc        0.4015  0.0
company_BA         0.4523  0.0
company_BA_argmax  0.5201  0.0
company_Precision  0.3630  0.0
company_Recall     0.4523  0.0
company_F1         0.4005  0.0
company_F1w        0.6174  0.0
company_Acc        0.7020  0.0
company_LogLoss    1.1691  0.0
company_ECE        0.2798  0.0
                     mean  std
model_BA           0.3019  0.0
model_Precision    0.3064  0.0
model_Recall       0.3019  0.0
model_F1           0.2925  0.0
model_F1w          0.2925  0.0
model_Acc          0.3019  0.0
model_LogLoss      2.3255  0.0
model_ECE          0.1487  0.0
sibling_Acc        0.4081  0.0
company_BA         0.4362  0.0
company_BA_argmax  0.5565  0.0
company_Precision  0.3525  0.0
company_

In [1]:
print(bootstrap_ci(base).round(4))
print(bootstrap_ci(base_test).round(4))

for lv in ['company']:
    print(pd.Series(np.diag(confusion(base, level=lv)), index=COMPANIES).round(4))

pred_c = M2C[base['P'][0].argmax(1)]
yc = M2C[base['y']]
print('1위 모델 기준 recall:', pd.Series([(pred_c[yc == c] == c).mean() for c in range(6)], index=COMPANIES).round(4).to_dict())
print('1위 모델 기준 Accuracy:', round((pred_c == yc).mean(), 4))

NameError: name 'bootstrap_ci' is not defined

In [2]:
from google.colab import drive
drive.mount('/content/drive')

def path(name):
    return f'/content/drive/MyDrive/{name}'

Mounted at /content/drive


In [3]:
data = load_data(path('train.csv'), path('val.csv'), path('test.csv'))
base = load_run(path('val_BASE_tfidfchar_v2.csv'), data)
base_test = load_run(path('test_BASE_tfidfchar_v2.csv'), data, split='test', open_test=True)

NameError: name 'load_data' is not defined

In [4]:
# ===== 1. Drive 연결 + 경로 =====
from google.colab import drive
drive.mount('/content/drive')

def path(name):
    return f'/content/drive/MyDrive/{name}'

# ===== 2. 공통 평가 코드 =====
import numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, log_loss,
                             confusion_matrix, precision_score, recall_score)

CLASSES = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
           'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
           'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
           'gpt-6-astra','kimi-k3','qwen3.8-max']
PREFIX = {'claude':'Anthropic','gemini':'Google','glm':'ZAI','gpt':'OpenAI','kimi':'Moonshot','qwen':'Qwen'}
COMPANIES = sorted(PREFIX.values())
M2C = np.array([COMPANIES.index(PREFIX[m.split('-')[0].rstrip('0123456789.')]) for m in CLASSES])
MULTI = np.bincount(M2C, minlength=6) > 1

def to_company(P):
    Q = np.zeros((len(P), 6))
    for m, c in enumerate(M2C):
        Q[:, c] += P[:, m]
    return Q

def ece(y, P, n_bins=15):
    conf, hit = P.max(1), P.argmax(1) == y
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    return float(sum((bins == b).mean() * abs(hit[bins == b].mean() - conf[bins == b].mean())
                     for b in range(n_bins) if (bins == b).any()))

def score(y, P):
    pred, yc, Pc = P.argmax(1), M2C[y], to_company(P)
    predc = Pc.argmax(1)
    mask = MULTI[yc] & (M2C[pred] == yc)
    return {
        'model_BA': balanced_accuracy_score(y, pred),
        'model_Precision': precision_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_Recall': recall_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_F1': f1_score(y, pred, average='macro', labels=range(18), zero_division=0),
        'model_F1w': f1_score(y, pred, average='weighted', labels=range(18), zero_division=0),
        'model_Acc': accuracy_score(y, pred),
        'model_LogLoss': log_loss(y, P, labels=range(18)),
        'model_ECE': ece(y, P),
        'sibling_Acc': (pred[mask] == y[mask]).mean() if mask.any() else np.nan,
        'company_BA': balanced_accuracy_score(yc, predc),
        'company_BA_argmax': balanced_accuracy_score(yc, M2C[pred]),
        'company_Precision': precision_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_Recall': recall_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_F1': f1_score(yc, predc, average='macro', labels=range(6), zero_division=0),
        'company_F1w': f1_score(yc, predc, average='weighted', labels=range(6), zero_division=0),
        'company_Acc': accuracy_score(yc, predc),
        'company_LogLoss': log_loss(yc, Pc, labels=range(6)),
        'company_ECE': ece(yc, Pc),
    }

def load_data(train_path, val_path, test_path):
    parts = {'train': train_path, 'val': val_path, 'test': test_path}
    data = pd.concat([pd.read_csv(p).assign(split=k) for k, p in parts.items()], ignore_index=True)
    n = data.split.value_counts().to_dict()
    assert n == {'train': 1889, 'test': 414, 'val': 396}, f'A의 분할 파일이 아님: {n}'
    assert data.id.is_unique and data.groupby('prompt_id').split.nunique().eq(1).all(), '분할 오류'
    return data[['id', 'prompt_id', 'model', 'split']]

def load_run(paths, data, split='val', open_test=False):
    if split not in ('val', 'test'):
        raise ValueError("split은 'val' 또는 'test'만 가능")
    if split == 'test' and not open_test:
        raise PermissionError('test는 정해진 시점에만 open_test=True로 연다')
    paths = [paths] if isinstance(paths, str) else list(paths)
    meta = data[data.split == split].sort_values('id').reset_index(drop=True)
    Ps = []
    for p in paths:
        df = pd.read_csv(p)
        assert list(df.columns) == ['id'] + CLASSES, f'{p}: 컬럼은 id + CLASSES(알파벳순)'
        assert df.id.is_unique and set(df.id) == set(meta.id), f'{p}: id가 평가 대상과 불일치'
        P = df.set_index('id').loc[meta.id, CLASSES].to_numpy(float)
        assert np.isfinite(P).all() and (P >= 0).all(), f'{p}: NaN 또는 음수 확률'
        assert np.allclose(P.sum(1), 1, atol=1e-3), f'{p}: 행 합이 1이 아님'
        Ps.append(P / P.sum(1, keepdims=True))
    y = meta.model.map({m: i for i, m in enumerate(CLASSES)}).to_numpy()
    return {'split': split, 'meta': meta, 'P': np.stack(Ps), 'y': y}

def evaluate(run):
    rows = pd.DataFrame([score(run['y'], P) for P in run['P']])
    return pd.DataFrame({'mean': rows.mean(), 'std': rows.std(ddof=1) if len(rows) > 1 else 0.0})

def _boot(meta, n_boot, seed):
    rng = np.random.default_rng(seed)
    groups = list(meta.groupby('prompt_id').indices.values())
    for _ in range(n_boot):
        yield np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])

def _ba(y, pred, n):
    tot = np.bincount(y, minlength=n)
    hit = np.bincount(y, weights=pred == y, minlength=n)
    return (hit[tot > 0] / tot[tot > 0]).mean()

def _fast(y, Ps):
    return np.mean([[_ba(y, P.argmax(1), 18), _ba(M2C[y], to_company(P).argmax(1), 6)] for P in Ps], 0)

def bootstrap_ci(run, n_boot=1000, seed=0):
    b = np.array([_fast(run['y'][i], run['P'][:, i]) for i in _boot(run['meta'], n_boot, seed)])
    lo, hi = np.percentile(b, [2.5, 97.5], axis=0)
    return pd.DataFrame({'ci_low': lo, 'ci_high': hi}, index=['model_BA', 'company_BA'])

def compare(base, new, n_boot=1000, seed=0):
    assert base['split'] == new['split'] and base['meta'].id.equals(new['meta'].id), '평가 대상이 다름'
    d = np.array([_fast(base['y'][i], new['P'][:, i]) - _fast(base['y'][i], base['P'][:, i])
                  for i in _boot(base['meta'], n_boot, seed)])
    lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
    out = pd.DataFrame({'diff_mean': d.mean(0), 'ci_low': lo, 'ci_high': hi}, index=['model_BA', 'company_BA'])
    out['improved'] = out.ci_low > 0
    out['worse'] = out.ci_high < 0
    return out

def confusion(run, level='model'):
    y, names = (run['y'], CLASSES) if level == 'model' else (M2C[run['y']], COMPANIES)
    pred = (lambda P: P.argmax(1)) if level == 'model' else (lambda P: to_company(P).argmax(1))
    Ms = [confusion_matrix(y, pred(P), labels=range(len(names)), normalize='true') for P in run['P']]
    return pd.DataFrame(np.mean(Ms, 0), index=names, columns=names)

def leaderboard(runs):
    return pd.DataFrame({k: evaluate(r).apply(lambda s: f"{s['mean']:.4f} ± {s['std']:.4f}", axis=1)
                         for k, r in runs.items()}).T

def template_table(val_run, test_run, level='model'):
    rows = {'Accuracy': 'Acc', 'Precision (Macro)': 'Precision', 'Recall (Macro) = BA': 'Recall',
            'F1-score (Macro)': 'F1', 'F1-score (Weighted)': 'F1w', 'Log loss': 'LogLoss', 'ECE': 'ECE'}
    if level == 'model':
        rows['형제 정확도'] = None
    dv, te = evaluate(val_run), evaluate(test_run)
    key = lambda v: 'sibling_Acc' if v is None else f'{level}_{v}'
    return pd.DataFrame({'Validation': [f"{dv.loc[key(v), 'mean']:.4f} ± {dv.loc[key(v), 'std']:.4f}" for v in rows.values()],
                         'Test': [f"{te.loc[key(v), 'mean']:.4f}" for v in rows.values()]}, index=list(rows))

# ===== 3. 파일 불러오기 =====
data = load_data(path('train.csv'), path('val.csv'), path('test.csv'))
base = load_run(path('val_BASE_tfidfchar_v2.csv'), data)
base_test = load_run(path('test_BASE_tfidfchar_v2.csv'), data, split='test', open_test=True)

# ===== 4. 문서 숫자 확인 =====
print('=== 95% 신뢰구간 ===')
print('val'); print(bootstrap_ci(base).round(4))
print('test'); print(bootstrap_ci(base_test).round(4))

print('\n=== val 개발사별 합산 recall ===')
print(pd.Series(np.diag(confusion(base, level='company')), index=COMPANIES).round(4))

pred_c, yc = M2C[base['P'][0].argmax(1)], M2C[base['y']]
print('\n=== val 1위 모델 기준 ===')
print('recall:', pd.Series([(pred_c[yc == c] == c).mean() for c in range(6)], index=COMPANIES).round(4).to_dict())
print('Accuracy:', round((pred_c == yc).mean(), 4))

# ===== 5. 결과 파일 저장 (내 드라이브) =====
metrics = pd.concat({'val': evaluate(base)['mean'], 'test': evaluate(base_test)['mean']}, axis=1).round(4)
metrics.to_csv(path('baseline_v2_metrics.csv'))
pd.concat({'val': bootstrap_ci(base), 'test': bootstrap_ci(base_test)}).round(4).to_csv(path('baseline_v2_ci.csv'))
template_table(base, base_test, level='model').to_csv(path('baseline_v2_table_model.csv'))
template_table(base, base_test, level='company').to_csv(path('baseline_v2_table_company.csv'))
print('\n결과 파일 4개 저장 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 95% 신뢰구간 ===
val
            ci_low  ci_high
model_BA    0.2374   0.3434
company_BA  0.4295   0.4712
test
            ci_low  ci_high
model_BA    0.2609   0.3406
company_BA  0.4112   0.4587

=== val 개발사별 합산 recall ===
Anthropic    0.8273
Google       0.9318
Moonshot     0.0000
OpenAI       0.9545
Qwen         0.0000
ZAI          0.0000
dtype: float64

=== val 1위 모델 기준 ===
recall: {'Anthropic': 0.7091, 'Google': 0.9432, 'Moonshot': 0.0909, 'OpenAI': 0.8545, 'Qwen': 0.3182, 'ZAI': 0.2045}
Accuracy: 0.6894

결과 파일 4개 저장 완료
